In [0]:
# Notebook: 05_generate_deliveries
# Purpose: Simulate delivery events linked to orders for ZapFlow platform
# Layer: Simulation

import random
from datetime import datetime, timedelta
from pyspark.sql.types import *

# ── Configuration ─────────────────────────────────────────────
OUTPUT_PATH  = "/Volumes/zapflow_raw/raw_data/raw_files/deliveries/"
ORDERS_PATH  = "/Volumes/zapflow_raw/raw_data/raw_files/orders/"
SEED = 42
random.seed(SEED)

# ── Delivery partners ──────────────────────────────────────────
NUM_DELIVERY_PARTNERS = 200
PARTNER_IDS = [f"DP_{i:05d}" for i in range(1, NUM_DELIVERY_PARTNERS + 1)]

# ── Realistic delivery patterns ────────────────────────────────
# Quick commerce SLA = 10 minutes
# Most deliveries happen within SLA
# Some partners are consistently late — realistic behavior

# Partners 1-150: reliable (mostly on time)
# Partners 151-200: unreliable (frequently late)
RELIABLE_PARTNERS   = PARTNER_IDS[:150]
UNRELIABLE_PARTNERS = PARTNER_IDS[150:]

DELIVERY_STATUSES = ["delivered", "failed", "returned_to_store"]

# ── Helper ─────────────────────────────────────────────────────
def get_delivery_minutes(partner_id):
    """
    Reliable partners: mostly 7-12 mins
    Unreliable partners: mostly 13-25 mins
    Small % of extreme outliers for both
    """
    if partner_id in RELIABLE_PARTNERS:
        return random.choices(
            [random.randint(7, 10),   # within SLA
             random.randint(11, 15),  # slightly late
             random.randint(16, 30)], # very late
            weights=[75, 20, 5]
        )[0]
    else:
        return random.choices(
            [random.randint(7, 10),   # within SLA
             random.randint(11, 15),  # slightly late
             random.randint(16, 30)], # very late
            weights=[30, 40, 30]
        )[0]

def introduce_data_quality_issues(record, counter):
    issue_type = None
    if counter % 55 == 0:
        record["delivery_partner_id"] = None
        issue_type = "missing_partner_id"
    elif counter % 90 == 0:
        record["delivery_minutes"] = -5
        issue_type = "negative_delivery_time"
    record["_data_quality_issue"] = issue_type
    return record

# ── Generate records ───────────────────────────────────────────
def generate_deliveries():
    # Read orders we already generated
    df_orders = spark.read.option("header", "true").csv(ORDERS_PATH)

    # Only delivered or returned orders get a delivery record
    df_eligible = df_orders.filter(
        df_orders.order_status.isin(["delivered", "returned"])
    )

    order_rows = df_eligible.select("order_id", "order_timestamp", "store_id").collect()

    records    = []
    counter    = 1

    for row in order_rows:
        order_id        = row["order_id"]
        order_timestamp = datetime.strptime(row["order_timestamp"], "%Y-%m-%d %H:%M:%S")
        partner_id      = random.choice(PARTNER_IDS)
        delivery_mins   = get_delivery_minutes(partner_id)

        # Pickup happens 2-4 mins after order
        pickup_time   = order_timestamp + timedelta(minutes=random.randint(2, 4))
        delivered_time = pickup_time + timedelta(minutes=delivery_mins)

        is_sla_breached = delivery_mins > 10

        # Determine delivery status
        if is_sla_breached and random.random() < 0.05:
            status = "failed"
        elif random.random() < 0.03:
            status = "returned_to_store"
        else:
            status = "delivered"

        record = {
            "delivery_id":          f"DEL_{counter:08d}",
            "order_id":             order_id,
            "delivery_partner_id":  partner_id,
            "store_id":             row["store_id"],
            "pickup_timestamp":     str(pickup_time),
            "delivered_timestamp":  str(delivered_time),
            "delivery_minutes":     delivery_mins,
            "is_sla_breached":      is_sla_breached,
            "delivery_status":      status,
            "_data_quality_issue":  None
        }

        record = introduce_data_quality_issues(record, counter)
        records.append(record)
        counter += 1

    return records

# ── Main execution ─────────────────────────────────────────────
print("Generating delivery data...")
print("Reading orders and generating linked delivery records...")

delivery_records = generate_deliveries()

schema = StructType([
    StructField("delivery_id",          StringType(),  False),
    StructField("order_id",             StringType(),  True),
    StructField("delivery_partner_id",  StringType(),  True),
    StructField("store_id",             StringType(),  True),
    StructField("pickup_timestamp",     StringType(),  True),
    StructField("delivered_timestamp",  StringType(),  True),
    StructField("delivery_minutes",     IntegerType(), True),
    StructField("is_sla_breached",      BooleanType(), True),
    StructField("delivery_status",      StringType(),  True),
    StructField("_data_quality_issue",  StringType(),  True),
])

df_deliveries = spark.createDataFrame(delivery_records, schema=schema)

# ── Save to Volume ─────────────────────────────────────────────
(df_deliveries
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(OUTPUT_PATH))

print(f"✅ Generated {len(delivery_records)} delivery records")
print(f"✅ Saved to {OUTPUT_PATH}")

# ── Sanity checks ──────────────────────────────────────────────
print("\n── Delivery status distribution ──")
df_deliveries.groupBy("delivery_status").count().orderBy("count", ascending=False).show()

print("── SLA breach summary ──")
df_deliveries.groupBy("is_sla_breached").count().show()

print("── Data quality issues ──")
df_deliveries.groupBy("_data_quality_issue").count().show()

print("── Sample records ──")
df_deliveries.show(5, truncate=False)